In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import json
import os

In [ ]:
def get_results_by_subset(path):
    models_results = [folder for folder in os.listdir(path) if "model" in folder]

    results_by_subset = defaultdict(list)
    for model in models_results:
        jsons = os.listdir(f"{path}/{model}")
        results = [json.load(open(f"{path}/{model}/{file}", "r")) for file in jsons]
        for result in results:
            results_by_subset[f"{result['dataset']['subset']}"].append(result)

    results_by_subset = dict(results_by_subset)
    return dict(sorted(results_by_subset.items()))


def get_avg_accs(results_by_subset):
    pos_accs = defaultdict(list)

    for _, results in results_by_subset.items():
        days = [value for value in results[0]["model"].keys() if "days" in value]

        for result in results:
            for day in days:
                pos_accs[day].append(result["model"][day]["pos_refinement"]["accuracy"])

    avg_pos_acc = []
    for day in days:
        avg_pos_acc.append(sum(pos_accs[day]) / len(pos_accs[day]))
    
    return avg_pos_acc


def get_accs_by_subset(results_by_subset, subset):
    results = results_by_subset[subset]
    days = [value for value in results[0]["model"].keys() if "days" in value]

    avg_pos_acc = []
    for day in days:
        pos_accs = [result["model"][day]["pos_refinement"]["accuracy"] for result in results]
        avg_pos_acc.append(sum(pos_accs) / len(pos_accs))
    
    return avg_pos_acc

def print_graph(accuracies):
    days = list(range(1, len(accuracies) + 1))

    plt.figure(figsize=(10, 5))
    plt.plot(days, accuracies, marker='o', linestyle='-', color='b', label='Accuracy')

    plt.title('Accuracy over Days')
    plt.xlabel('Days')
    plt.ylabel('Accuracy')

    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.xticks(days)

    for index, value in enumerate(accuracies):
        plt.text(days[index], value, f'{value:.4f}', ha='center', va='bottom')

    plt.show()

In [ ]:
RESULTS_PATH = "_results/exp_d88bbb7e-f412-4e40-8276-9cca9c910245"
results_by_subset = get_results_by_subset(RESULTS_PATH)

# Global results

In [ ]:
all_accuracies = get_avg_accs(results_by_subset)
print_graph(all_accuracies)

# Camera results

In [ ]:
SUBSET = "CNR-CAMERA-9"

camera_accurarices = get_accs_by_subset(results_by_subset, SUBSET)
print_graph(camera_accurarices)